# PPTX Translation Workflow
Extract and review source text, translate with Gemini, evaluate with OpenAI, optionally use Claude for rejected translations, then finalize and reconstruct offline.

Developed and validated primarily for Biblical educational presentations. The evaluation and Claude fallback prompts include Protestant/Evangelical domain instructions; changing Gemini's context does not make those stages domain-neutral.


## Setup
Use Python with Jupyter and `python-pptx>=1.0.2,<2` (see [README](README.md) and `requirements.txt`). Live stages additionally require `google-genai`, `openai`, or optional `anthropic`, with `GEMINI_API_KEY`, `OPENAI_API_KEY`, or `ANTHROPIC_API_KEY` respectively. Supply credentials through the kernel environment; the notebook neither installs packages nor loads `.env` files. Offline extraction and previews need no provider credentials.

For this notebook's standard Gemini client, set **`GEMINI_API_KEY` only**. The [Google Gen AI SDK](https://googleapis.github.io/python-genai/) can discover either `GOOGLE_API_KEY` or `GEMINI_API_KEY` when no key is passed explicitly, but this workflow checks `GEMINI_API_KEY` and passes it explicitly. `GOOGLE_API_KEY` alone does not satisfy that check; there is no need to set both.

The checkout may have any directory name. Launch from its root, or its parent when exactly one matching checkout is present. From elsewhere or when discovery is ambiguous, set `PPTX_WORKFLOW_ROOT` to the project directory before starting the kernel. Notebook paths below are based on `project_root`.


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
import json
import hashlib
import os

# Launch from the project root or its parent, or configure an exact root.
# PPTX_V3_ROOT remains supported for existing launch configurations.
configured_root = os.environ.get("PPTX_WORKFLOW_ROOT", os.environ.get("PPTX_V3_ROOT"))
project_root = Path(configured_root if configured_root is not None else Path.cwd()).expanduser().resolve()
if configured_root is None and not (project_root / "pptx_translation" / "workflow_config.py").is_file():
    root_candidates = sorted(
        path.parent.parent for path in project_root.glob("*/pptx_translation/workflow_config.py")
        if path.is_file()
    )
    if len(root_candidates) != 1:
        raise RuntimeError("Set PPTX_WORKFLOW_ROOT to the project directory; no unique checkout was found here.")
    project_root = root_candidates[0].resolve()
if not (project_root / "pptx_translation" / "workflow_config.py").is_file():
    raise RuntimeError("Set PPTX_WORKFLOW_ROOT to the project directory containing pptx_translation.")
sys.path.insert(0, str(project_root))
from pptx_translation import pptx_helpers as ph
from pptx_translation.workflow_config import WorkflowConfig, LazyAPIClients, api_key_status


## Configuration
Replace the saved project-specific `input_path` with your presentation; choose source/target languages and keep outputs separate from inputs. `include_smartart` controls SmartArt extraction. Styled-span preservation is always enabled. `write_reports=True` saves optional extraction/QC details inside the selected run, or defers them until a fresh run is created.

`output_root` can optionally point to a separate directory for runs, checkpoints, reports and deliverables, such as `Path(r"C:\llm_translation_outputs\pptx")` on Windows. A nonsynchronized local directory may reduce checkpoint-write interference when the checkout is in OneDrive or another cloud-synchronized folder. Resume uses the selected manifest's existing location; changing `output_root` does not relocate saved runs.

For a fresh run, leave `run_manifest_path=None`. For an existing run, select its exact `run.json`; do not edit derived stage checkpoint variables. First use Run All with all six execution flags `False` to inspect extraction and stage previews without API calls or exports. Each stage must then be enabled explicitly; no latest-run discovery or automatic stage execution occurs.

`primary_domain_context` optionally supplements Gemini's base instruction. It affects request identity and must stay compatible when resuming; it does not change evaluator or fallback prompts.


In [ ]:
run_manifest_path = None
# None for Fresh run; or uncomment and paste ONE exact run.json path here to resume an existing run.

# run_manifest_path = (
#     project_root
#     / "outputs"
#     / "runs"
#     / "<insert manifest directory name here>"
#     / "run.json"
# )

# Styled-span preservation is always enabled in the production workflow.

config = WorkflowConfig(
    input_path=project_root / "pptx_files" / "Help_I_Dont_Understand_The_Bible.pptx",
    output_root=project_root / "outputs",
    source_language="English",
    target_language="Traditional Chinese",
    include_smartart=True,
    write_reports=False,
)
print("Credential status:", api_key_status())
available, version = ph.check_python_pptx_available()
if not available:
    raise ImportError("python-pptx is required. See README; no packages were installed.")
print("python-pptx:", version)
if not config.input_path.exists():
    if config.input_path == project_root / "examples" / "stage1_sample.pptx":
        from examples.create_sample import create_sample
        create_sample(config.input_path)
    else:
        raise FileNotFoundError(config.input_path)

from pptx_translation.active_run import ActiveRun
active_run = None
if run_manifest_path is not None:
    selected_manifest = Path(run_manifest_path).expanduser()
    if not selected_manifest.is_absolute():
        selected_manifest = project_root / selected_manifest
    active_run = ActiveRun.open(selected_manifest, config, styled=True)
    print("Selected run:", active_run.state["run_id"], "| manifest:", active_run.manifest_path)


In [ ]:
# # Optional Gemini primary translation context (supplementary to the base instruction).
# primary_domain_context = None

# Example:
primary_domain_context = (
    "This presentation is part of a Christian Bible-study workshop series "
    "designed to help people who feel frustrated or uncertain about "
    "understanding the Bible. The tone is approachable and encouraging, "
    "with occasional lighthearted humor. Preserve the intended tone "
    "without introducing humor where none exists."
)
print(primary_domain_context)

## Extraction
Extract paragraph-level units from text boxes, grouped shapes, table cells and, when enabled, SmartArt data. Retain source locators, formatting metadata and ordered styled spans. Translate each paragraph in context; text spans map wording back to formatting boundaries, while structural spans preserve items such as tabs and soft breaks locally.

SmartArt extraction does not guarantee editable reconstruction. Native data/cache synchronization supports conservatively validated **IconCircleLabelList** diagrams; unsupported layouts or text structures require manual review.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_hash = sha256(config.input_path)
presentation, presentation_summary = ph.open_presentation(config.input_path)
inspection = ph.inspect_presentation_structure(presentation)
units = ph.build_pptx_text_units(presentation, pptx_path=config.input_path,
                                include_smartart=config.include_smartart,
                                include_styled_spans=True)
extraction_summary = ph.summarize_pptx_text_units(units)
print("Extracted units:", extraction_summary["total_text_units"], "| source types:", extraction_summary["text_units_by_source_type"])


## Classification
Review which units require translation and which retain source text. Classification recognizes Bible references and numeric/preserved content. Inspect the samples and preserved classifications before translating.


In [ ]:
classified_units = ph.classify_pptx_text_units(units)
classification_summary = ph.summarize_pptx_classification(classified_units)
print("Classification | total:", classification_summary["total_units"], "| translate:", classification_summary["translate_units"], "| preserve:", classification_summary["preserve_units"])
ph.build_classification_sample_rows(classified_units, sample_size=2)


## Quality checks
Review extraction counts, fragmented text, duplicates and flagged shapes; inspect `qc_reports` for details beyond the printed totals. Charts, picture-embedded text, notes, masters and layouts are outside text extraction and need manual inspection. QC does not establish translation or visual completeness.


In [ ]:
qc_summary = ph.build_pretranslation_qc_summary(inspection, units)
qc_reports = {
    "tables": ph.build_table_audit_rows(units),
    "source_type_samples": ph.build_source_type_samples(units),
    "per_slide": ph.build_per_slide_extraction_report(inspection, units),
    "empty_text_frames": ph.build_empty_text_frame_report(inspection),
    "long_text": ph.build_long_text_unit_report(units),
    "fragmented_runs": ph.build_run_fragmentation_report(units),
    "duplicate_text": ph.build_duplicate_text_report(units),
    "preserve_candidates": ph.build_preserve_candidate_report(units),
    "manual_shapes": ph.build_manual_shape_review_rows(inspection),
    "paragraph_structure": ph.inspect_text_unit_paragraph_structure(presentation, units),
    "run_details": ph.inspect_text_unit_runs(presentation, units),
}
assert sha256(config.input_path) == source_hash, "Input changed during inspection."
print("QC:", {key: value for key, value in qc_summary.items() if key.endswith("_count")})
print("Extraction count difference:", qc_summary["count_reconciliation"]["difference"], "| full details: qc_reports")
production_qc_report = None
qc_report_path = None
if active_run is not None and active_run.state.get("artifact_layout") == 1:
    saved_qc_path = active_run.manifest_path.parent / "reports" / "extraction_qc.json"
    if saved_qc_path.is_file():
        qc_report_path = saved_qc_path.resolve()
if config.write_reports:
    production_qc_report = {"source_sha256": source_hash, "source": str(config.input_path),
              "source_language": config.source_language, "target_language": config.target_language,
              "presentation": presentation_summary, "classification": classification_summary,
              "qc": qc_summary, "reports": qc_reports, "units": classified_units}
    if active_run is not None and active_run.state.get("artifact_layout") == 1:
        from pptx_translation.artifact_layout import write_qc_report
        qc_report_path = write_qc_report(active_run, production_qc_report).resolve()
    else:
        print("Production QC export deferred until a new translation run is created.")
if qc_report_path is not None:
    print("Saved QC report:", qc_report_path)
else:
    print("QC details are in memory only: qc_reports (no saved report).")


## Gemini settings
Confirm model access, temperature and batching settings before enabling execution. The saved `primary_batch_limit=None` allows all remaining eligible work. Set a nonnegative integer to limit workflow submissions, including retries, for this invocation; `0` prepares the checkpoint without submissions. This limit counts submissions, not paragraphs, and does not bound provider SDK retries.


In [ ]:
# Primary translation: prepare the COMPLETE plan, then submit at most N requests.
enable_primary_translation = False
primary_batch_limit = None  # Actual requests, including retries; None = all remaining.
primary_model_name = "gemini-3.1-pro-preview" # "gemini-3.6-flash"  # Previously tested model; confirm access before enabling.
primary_temperature = 1.0
primary_system_instruction = "Translate each complete paragraph into metadata.target_language, retaining full paragraph context and obeying response_contract and translation_note."
primary_max_units = 80
primary_max_chars = 12000



## Translation manifest
Review all eligible paragraphs and batch sizes before execution. Preserve source, languages, extraction settings, model, temperature, instruction, domain context and batching settings when resuming; incompatible changes require a fresh run. The submission limit may change.

To deliberately start another run, set `run_manifest_path=None` and rerun configuration before enabling Gemini. Rerunning only the Gemini execution cell continues the active run.


In [ ]:
from pptx_translation.primary_translation import build_primary_payload, batch_primary_payload, primary_batch_manifest, primary_validation_policy

primary_plan_policy = primary_validation_policy(active_run.path("primary") if globals().get("active_run") else None)
primary_payload = build_primary_payload(classified_units, config.source_language, config.target_language, policy_version=primary_plan_policy)
primary_batches = batch_primary_payload(primary_payload, max_units=primary_max_units, max_chars=primary_max_chars)
primary_manifest = primary_batch_manifest(primary_batches)
print("Primary plan:", len(primary_payload["units"]), "eligible units |", len(primary_batches), "batches")
print("Submission limit this invocation:", primary_batch_limit)
print("Plan detail remains available in primary_manifest and primary_batches.")
from pptx_translation.run_reporting import inspect_unit  # On demand: inspect_unit(active_run.manifest_path, "s001_u0001")
from importlib.metadata import version, PackageNotFoundError
try:
    print("Installed google-genai:", version("google-genai"))
except PackageNotFoundError:
    print("google-genai is not installed in this kernel; install/select the correct environment before a live test.")


## Run Gemini translation
Enable Gemini to start or continue the run, then disable its flag afterward. A fresh run creates and prints a unique manifest path; save that exact path. Validated paragraphs skip automatically and retry budgets persist.

After a kernel restart, select the saved `run_manifest_path`, keep source/languages/extraction and stage settings compatible, and Run All with flags off. Review the saved status, then enable and execute the required stage. The manifest supplies checkpoint handoffs and matching resume paths; it does not execute stages for you.


In [ ]:
from pptx_translation.primary_translation import run_primary_translation, inspect_primary_checkpoint

primary_result = None
if enable_primary_translation:
    if not primary_model_name.strip():
        raise ValueError("Set primary_model_name to an available Gemini model ID before execution.")
    if globals().get("active_run") is None:
        active_run = ActiveRun.create(config, classified_units, styled=True)
        run_manifest_path = active_run.manifest_path
    if globals().get("production_qc_report") is not None and active_run.state.get("artifact_layout") == 1:
        from pptx_translation.artifact_layout import write_qc_report
        qc_report_path = write_qc_report(active_run, production_qc_report).resolve()
        print("Saved QC report:", qc_report_path)
    primary_checkpoint, primary_resume = active_run.prepare("primary")
    print("Active run manifest (save this path for restart):", active_run.manifest_path)
    primary_source_units = active_run.primary_units(classified_units)
    primary_clients = LazyAPIClients(replace(config, enable_paid_api=True))  # Internal gate follows this stage flag.
    active_primary_client = globals().get("primary_client")
    if active_primary_client is None:
        active_primary_client = primary_clients.get("gemini")  # Gated SDK construction; no key is printed.
    print("Primary checkpoint:", primary_checkpoint)
    try:
        primary_result = run_primary_translation(
            active_primary_client, primary_source_units, config.input_path, config.source_language,
            config.target_language, primary_model_name, primary_system_instruction, primary_checkpoint,
            resume=primary_resume,
            temperature=primary_temperature,
            domain_context=primary_domain_context,
            max_units=primary_max_units, max_chars=primary_max_chars, max_batches=primary_batch_limit,
        )
    finally:
        if primary_checkpoint.exists():
            active_run.record("primary", primary_checkpoint)

else:
    if globals().get("active_run") and active_run.path("primary"):
        primary_result = inspect_primary_checkpoint(active_run.path("primary"))
    print("Primary execution disabled; showing the selected run or prepared plan without API calls.")


## Primary results
Completed means structurally validated, not semantically accepted. Each paragraph has at most three recorded submission attempts; retries include actionable validation feedback and never resubmit validated units. Review untouched, retry-pending and exhausted units separately. Provider/validation errors continue to later batches; integrity or persistence errors stop the stage. Recorded attempts are not guaranteed network-call counts, especially after interruptions or SDK retries.

For saved unit details, use `inspect_unit(active_run.manifest_path, "s001_u0001")`. Result objects and batch plans remain available; normal output shows compact counts and a few exception cases.


In [ ]:
if primary_result is not None:
    completed_keys = set(primary_result["completed_unit_keys"])
    failed_keys = set(primary_result["error_unit_keys"])
    pending_keys = set(primary_result["pending_unit_keys"])
    retry_pending_keys = set(primary_result["retry_pending_unit_keys"])
    exhausted_keys = set(primary_result["exhausted_unit_keys"])
    from pptx_translation.run_reporting import print_stage_summary
    print_stage_summary(globals().get("active_run"), "primary")


## OpenAI primary evaluation
Finish Gemini processing first. The selected run supplies its primary checkpoint; only structurally validated candidates are evaluated. OpenAI returns one **binary accept/reject decision per complete paragraph**, using styled spans as context rather than separate evaluation targets. It does not rewrite translations.

Acceptance (`passed=true`) has empty feedback. Rejection (`passed=false`) requires a short, concrete reason for a substantive meaning, formatting or literal-content problem. Faithful wording variation is allowed. Numeric warnings are observations, not automatic failures; the evaluator checks numerical and reference meaning. Python separately validates structural formatting.


In [ ]:
# Evaluation reads completed primary translations; it never retranslates them.
enable_evaluation = False
evaluation_batch_limit = None  # None = all remaining; 0 = save plan without submissions.
evaluation_model_name = "gpt-6.1-sol"  # Saved evaluator model; select an accessible model if needed.

# Automatic handoffs from the explicitly selected run.
evaluation_primary_checkpoint = (active_run.path("primary") if globals().get("active_run") else None)


In [ ]:
from pptx_translation.evaluation import (build_evaluation_payload, batch_evaluation_payload, run_evaluation,
                        EvaluationStageError, inspect_evaluation_checkpoint)

evaluation_payload = None
evaluation_batches = []
evaluation_primary_path = Path(evaluation_primary_checkpoint).expanduser() if evaluation_primary_checkpoint else None
if evaluation_primary_path is not None:
    if not evaluation_primary_path.is_absolute():
        evaluation_primary_path = project_root / evaluation_primary_path
    try:
        evaluation_payload = build_evaluation_payload(evaluation_primary_path, config.input_path)
    except (ValueError, OSError) as exc:
        if enable_evaluation:
            raise
        print("Evaluation plan unavailable for selected source/checkpoint:", str(exc))
if evaluation_payload is not None:
    evaluation_batches = batch_evaluation_payload(evaluation_payload)
    print("Evaluation plan:", len(evaluation_payload["units"]), "completed primary units |",
          len(evaluation_batches), "evaluation batches | limit:", evaluation_batch_limit)
    print("Primary units not yet completed (excluded):", evaluation_payload["metadata"]["primary_uncompleted_units"])


## Run primary evaluation
Enable evaluation with `evaluation_batch_limit=None` for all remaining batches, a finite cap, or `0` to save the plan without submissions. Disable the flag afterward. Resume skips completed accept and reject decisions; operational errors retry only when you explicitly rerun the enabled stage. A rejection is a completed decision, not a provider error.

Changed upstream checkpoint hashes invalidate dependent handoffs while retaining their files. The next enabled downstream stage starts a compatible branch. Do not modify checkpoints to bypass compatibility checks.


In [ ]:
evaluation_result = None
if enable_evaluation:
    if globals().get("active_run") is None:
        raise ValueError("Select this run manifest before enabling evaluation.")
    evaluation_primary_path = active_run.path("primary")
    evaluation_payload = build_evaluation_payload(evaluation_primary_path, config.input_path) if evaluation_primary_path else None
    if evaluation_payload is None or not evaluation_payload["units"]:
        raise ValueError("Select a primary checkpoint with completed translations.")
    evaluation_clients = LazyAPIClients(replace(config, enable_paid_api=True))  # Internal gate follows this stage flag.
    active_evaluation_client = globals().get("evaluation_client")
    if active_evaluation_client is None:
        active_evaluation_client = evaluation_clients.get("openai")
    evaluation_checkpoint, stage_resume = active_run.prepare("evaluation")
    print("Evaluation checkpoint:", evaluation_checkpoint)
    try:
        evaluation_result = run_evaluation(
            active_evaluation_client, evaluation_primary_path, config.input_path,
            evaluation_checkpoint, evaluation_model_name,
            resume=stage_resume, max_batches=evaluation_batch_limit)
    except EvaluationStageError as exc:
        print(str(exc))
        evaluation_result = inspect_evaluation_checkpoint(evaluation_checkpoint)
    finally:
        if evaluation_checkpoint.exists():
            active_run.record("evaluation", evaluation_checkpoint)
else:
    print("OpenAI evaluation disabled; no provider client or requests created.")

from pptx_translation.run_reporting import print_stage_summary
print_stage_summary(globals().get("active_run"), "evaluation")


## Claude fallback
Claude is optional. Only explicit primary semantic rejections qualify; primary generation/validation failures and missing or failed evaluations do not. The selected run supplies the compatible primary/evaluation pair.

Claude receives the source paragraph, rejected candidate and evaluator feedback. Its candidates must pass structural validation and a separate OpenAI evaluation before finalization can select them. Enable this stage only when eligible rejections remain; otherwise skip both fallback stages.


In [ ]:
# Claude generation: independent execution flag and immutable upstream snapshots.
enable_fallback_translation = False
fallback_model_name = "claude-sonnet-5-5"  # Set an accessible Claude model before enabling.
fallback_batch_limit = None  # None = all remaining; 0 = save plan without submissions.

# Automatic handoffs from the explicitly selected run.
fallback_primary_checkpoint = (active_run.path("primary") if globals().get("active_run") else None)
fallback_evaluation_checkpoint = (active_run.path("evaluation") if globals().get("active_run") else None)


In [ ]:
from pptx_translation.fallback_translation import (build_fallback_payload, run_fallback_translation,
                                  inspect_fallback_checkpoint, FallbackStageError)

def fallback_input_path(value):
    if value is None:
        return None
    path = Path(value).expanduser()
    return path if path.is_absolute() else project_root / path

fallback_primary_path = fallback_input_path(fallback_primary_checkpoint)
fallback_evaluation_path = fallback_input_path(fallback_evaluation_checkpoint)
fallback_payload = None
if (fallback_primary_path is not None and fallback_evaluation_path is not None
        and fallback_primary_path.is_file() and fallback_evaluation_path.is_file()):
    try:
        fallback_payload = build_fallback_payload(fallback_primary_path, fallback_evaluation_path, config.input_path)
    except ValueError as exc:
        if enable_fallback_translation:
            raise
        print("Claude preview unavailable for selected source:", exc)
if fallback_payload is not None:
    print("Claude-eligible semantic rejections:", len(fallback_payload["units"]))
    print("Excluded counts:", {key: len(value) if isinstance(value, list) else value for key, value in fallback_payload["excluded"].items()})
    from pptx_translation.run_reporting import print_cases
    print_cases([f"{unit['unit_key']}: {unit['evaluator_feedback']}" for unit in fallback_payload["units"]])
else:
    print("Select existing primary and evaluation checkpoints to preview Claude eligibility.")


In [ ]:
fallback_result = None
if enable_fallback_translation:
    if globals().get("active_run") is None:
        raise ValueError("Select this run manifest before enabling Claude.")
    fallback_primary_path = active_run.path("primary")
    fallback_evaluation_path = active_run.path("evaluation")
    fallback_payload = (build_fallback_payload(fallback_primary_path, fallback_evaluation_path, config.input_path)
                        if fallback_primary_path and fallback_evaluation_path else None)
    if not fallback_model_name.strip():
        raise ValueError("Set fallback_model_name before enabling Claude generation.")
    if fallback_payload is None:
        raise ValueError("Select valid primary and evaluation checkpoints.")
    if not fallback_payload["units"]:
        print("No semantic rejections; no Claude client or requests created.")
    else:
        fallback_clients = LazyAPIClients(replace(config, enable_paid_api=True))
        active_fallback_client = globals().get("fallback_client")
        if active_fallback_client is None:
            active_fallback_client = fallback_clients.get("anthropic")
        fallback_checkpoint, stage_resume = active_run.prepare("fallback")
        print("Claude generation checkpoint:", fallback_checkpoint)
        try:
            fallback_result = run_fallback_translation(
                active_fallback_client, fallback_primary_path, fallback_evaluation_path,
                config.input_path, fallback_checkpoint, fallback_model_name,
                resume=stage_resume, max_batches=fallback_batch_limit)
        except FallbackStageError as exc:
            print(str(exc))
            fallback_result = inspect_fallback_checkpoint(fallback_checkpoint)
        finally:
            if fallback_checkpoint.exists():
                active_run.record("fallback", fallback_checkpoint)
else:
    print("Claude generation disabled; no provider client or requests created.")

from pptx_translation.run_reporting import print_stage_summary
print_stage_summary(globals().get("active_run"), "fallback")


Use `fallback_batch_limit` to control submissions for this invocation (`None` for all remaining, `0` for plan only). The enabled stage resumes its matching checkpoint; disable the flag after use. This is the initial Claude correction round, not an automatic loop over rejected fallback candidates.


## OpenAI fallback evaluation
Evaluate structurally validated Claude candidates using the same paragraph-level binary contract; OpenAI does not revise their text. Decisions bind to exact candidate text and styled-span responses. The model defaults to the recorded primary evaluator unless explicitly overridden.


In [ ]:
# Fallback evaluation: independent execution flag; selected Claude snapshot stays immutable.
enable_fallback_evaluation = False
fallback_evaluation_model_name = None  # Inherit the recorded primary evaluator; set a model string to override.
fallback_evaluation_batch_limit = None  # None = all remaining; 0 = save plan without submissions.

# Automatic handoffs from the explicitly selected run.
fallback_evaluation_input_checkpoint = (active_run.path("fallback") if globals().get("active_run") else None)


In [ ]:
from pptx_translation.fallback_evaluation import (build_fallback_evaluation_payload, run_fallback_evaluation,
                                 inspect_fallback_evaluation_checkpoint, resolve_evaluation_model)
from pptx_translation.evaluation import EvaluationStageError

fallback_evaluation_input_path = (Path(fallback_evaluation_input_checkpoint).expanduser()
                                  if fallback_evaluation_input_checkpoint is not None else None)
if fallback_evaluation_input_path is not None and not fallback_evaluation_input_path.is_absolute():
    fallback_evaluation_input_path = project_root / fallback_evaluation_input_path
fallback_evaluation_payload = None
if fallback_evaluation_input_path is None:
    print("Fallback evaluation not configured; select this run's Claude checkpoint when needed.")
else:
    try:
        fallback_evaluation_payload = build_fallback_evaluation_payload(
            fallback_evaluation_input_path, config.input_path)
        fallback_evaluation_preview_model = resolve_evaluation_model(
            fallback_evaluation_input_path, fallback_evaluation_model_name)
    except (ValueError, OSError) as exc:
        if enable_fallback_evaluation:
            raise
        fallback_evaluation_payload = None
        print("Fallback evaluation plan unavailable:", str(exc))
if fallback_evaluation_payload is not None:
    print("Fallback evaluator model:", fallback_evaluation_preview_model)
    print("Claude paragraphs ready for evaluation:", len(fallback_evaluation_payload["units"]))
    print("Excluded pending Claude units:", len(fallback_evaluation_payload["metadata"]["fallback_pending_unit_keys"]))
    print("Excluded failed Claude units:", len(fallback_evaluation_payload["metadata"]["fallback_error_unit_keys"]))


In [ ]:
fallback_evaluation_result = None
if enable_fallback_evaluation:
    if globals().get("active_run") is None:
        raise ValueError("Select this run manifest before enabling fallback evaluation.")
    fallback_evaluation_input_path = active_run.path("fallback")
    fallback_evaluation_payload = (build_fallback_evaluation_payload(fallback_evaluation_input_path, config.input_path)
                                   if fallback_evaluation_input_path else None)
    if fallback_evaluation_payload is None:
        raise ValueError("Select a valid Claude generation checkpoint.")
    if not fallback_evaluation_payload["units"]:
        print("No completed Claude candidates; no OpenAI client or requests created.")
    else:
        active_fallback_evaluation_model = resolve_evaluation_model(
            fallback_evaluation_input_path, fallback_evaluation_model_name)
        print("Fallback evaluator model:", active_fallback_evaluation_model)
        fallback_evaluation_clients = LazyAPIClients(replace(config, enable_paid_api=True))
        active_fallback_evaluation_client = globals().get("fallback_evaluation_client")
        if active_fallback_evaluation_client is None:
            active_fallback_evaluation_client = fallback_evaluation_clients.get("openai")
        fallback_evaluation_output, stage_resume = active_run.prepare("fallback_evaluation")
        print("Fallback evaluation checkpoint:", fallback_evaluation_output)
        try:
            fallback_evaluation_result = run_fallback_evaluation(
                active_fallback_evaluation_client, fallback_evaluation_input_path, config.input_path,
                fallback_evaluation_output, active_fallback_evaluation_model,
                resume=stage_resume,
                max_batches=fallback_evaluation_batch_limit)
        except EvaluationStageError as exc:
            print(str(exc))
            fallback_evaluation_result = inspect_fallback_evaluation_checkpoint(fallback_evaluation_output)
        finally:
            if fallback_evaluation_output.exists():
                active_run.record("fallback_evaluation", fallback_evaluation_output)
else:
    print("Fallback evaluation disabled; no provider client or requests created.")

from pptx_translation.run_reporting import print_stage_summary
print_stage_summary(globals().get("active_run"), "fallback_evaluation")


Enable fallback evaluation with the desired batch limit (`None` for all remaining, `0` for plan only), then disable its flag afterward. Matching completed decisions are reused; error batches retry on explicit rerun. A changed Claude checkpoint requires a new evaluation branch.


## Finalize translations and reports
Preview completeness, then enable offline export to reuse or create the authoritative JSON snapshot, CSV summary and Markdown follow-up report for this lineage. No PowerPoint is generated here.

Finalization selects primary candidates accepted by OpenAI, or accepted Claude candidates for rejected primary translations. Preserved units keep source text; unresolved units also keep source text and make coverage incomplete. An incomplete snapshot may still be exported and reconstructed as a partial translation.


In [ ]:
# Offline finalization: independent inputs; diagnostic fallback branches are rejected.
enable_finalization = False  # True exports or reuses this lineage; preview remains read-only.
finalization_output_root = (active_run.manifest_path.parent / ("checkpoints" if active_run.state.get("artifact_layout") == 1 else "finalization") if globals().get("active_run") else config.output_root / "finalization")

# Automatic handoffs from the explicitly selected run.
finalization_primary_checkpoint = (active_run.path("primary") if globals().get("active_run") else None)
finalization_evaluation_checkpoint = (active_run.path("evaluation") if globals().get("active_run") else None)
finalization_fallback_checkpoint = (active_run.path("fallback") if globals().get("active_run") else None)
finalization_fallback_evaluation_checkpoint = (active_run.path("fallback_evaluation") if globals().get("active_run") else None)


In [ ]:
from pptx_translation.finalization import build_finalization_snapshot, write_finalization_artifacts

def finalization_input_path(value):
    if value is None:
        return None
    path = Path(value).expanduser()
    return path if path.is_absolute() else project_root / path

finalization_snapshot = None
if finalization_primary_checkpoint is None or finalization_evaluation_checkpoint is None:
    if enable_finalization:
        raise ValueError("Select this run's primary and evaluation checkpoints before finalization.")
    print("Finalization not configured; no old snapshots selected.")
else:
    try:
        finalization_snapshot = build_finalization_snapshot(
            finalization_input_path(finalization_primary_checkpoint),
            finalization_input_path(finalization_evaluation_checkpoint), config.input_path,
            fallback_checkpoint=finalization_input_path(finalization_fallback_checkpoint),
            fallback_evaluation_checkpoint=finalization_input_path(finalization_fallback_evaluation_checkpoint))
    except (ValueError, OSError) as exc:
        if enable_finalization:
            raise
        print("Finalization preview unavailable:", str(exc))
if finalization_snapshot is not None:
    from pptx_translation.run_reporting import print_finalization_summary
    print_finalization_summary(finalization_snapshot, globals().get("active_run"))
    print("Preview only until export; unresolved units retain source text.")


In [ ]:
finalization_artifacts = None
if enable_finalization:
    if globals().get("active_run") is None:
        raise ValueError("Select this run manifest before exporting finalization.")
    finalization_artifacts = active_run.finalize()
    print("Finalization artifacts (reused when lineage is unchanged):")
    for name, path in finalization_artifacts.items():
        print(name, ":", path)
else:
    print("Finalization export disabled; preview only, no artifacts written.")
    from pptx_translation.run_reporting import print_stage_summary
    print_stage_summary(globals().get("active_run"), "finalization")


Every source unit is accounted for. Retained source-language text for unresolved work is not an accepted translation, and pending work is not automatically exhausted. `FLAGGED FOR MANUAL FOLLOWUP:` applies only to explicit reviewer-marked cases in reports; it does not add visible slide annotations or label every unresolved unit. The exported snapshot is registered for reconstruction automatically.


## Generate and validate PowerPoint
Use the run's saved finalization snapshot to reconstruct accepted candidates offline. Validation checks source hashes and unit identities, expected text and coverage, styled spans/run formatting, package relationships and unchanged non-target content, plus supported SmartArt data/cache consistency. It reopens the output to check text and source correspondence.

Compatible saved results are verified and reused without regeneration; failed attempts remain available for diagnosis. Structural validation, translation completeness and human visual/linguistic approval are separate: a structural pass does not prove that all content is translated or fits the slides.


In [ ]:
# Offline reconstruction: one write flag, selected immutable snapshot, new output directory.
enable_reconstruction = False
reconstruction_output_root = (active_run.manifest_path.parent / ("reports" if active_run.state.get("artifact_layout") == 1 else "translated_pptx") if globals().get("active_run") else config.output_root / "translated_pptx")

# Automatic handoffs from the explicitly selected run.
reconstruction_finalization_snapshot = (active_run.path("finalization") if globals().get("active_run") else None)


In [ ]:
from pptx_translation.finalized_reconstruction import build_reconstruction_plan, reconstruct_finalized_snapshot

reconstruction_snapshot_path = (Path(reconstruction_finalization_snapshot).expanduser()
                                if reconstruction_finalization_snapshot is not None else None)
if reconstruction_snapshot_path is not None and not reconstruction_snapshot_path.is_absolute():
    reconstruction_snapshot_path = project_root / reconstruction_snapshot_path
reconstruction_plan = None
if reconstruction_snapshot_path is None:
    if enable_reconstruction:
        raise ValueError("Select this run's saved finalization snapshot before reconstruction.")
    print("Reconstruction not configured; no old finalization snapshot selected.")
else:
    try:
        reconstruction_plan = build_reconstruction_plan(reconstruction_snapshot_path, config.input_path)
    except (ValueError, OSError) as exc:
        if enable_reconstruction:
            raise
        print("Reconstruction preview unavailable:", str(exc))
if reconstruction_plan is not None:
    print("Reconstruction coverage:", reconstruction_plan["snapshot"]["status"])
    print("Accepted targets:", len(reconstruction_plan["units"]),
          "| ordinary:", len(reconstruction_plan["paragraph_replacements"]),
          "| styled:", len(reconstruction_plan["span_replacements"]))
    print("Unresolved:", reconstruction_plan["snapshot"]["summary"]["unresolved"])


In [ ]:
finalized_reconstruction_result = None
if enable_reconstruction:
    if globals().get("active_run") is None:
        raise ValueError("Select this run manifest before reconstruction.")
    finalized_reconstruction_result = active_run.reconstruct()
    reconstruction_report = finalized_reconstruction_result["report"]
    from pptx_translation.run_reporting import print_reconstruction_summary
    print_reconstruction_summary(finalized_reconstruction_result, active_run)
    if reconstruction_report["status"] not in ("passed", "manual_review"):
        raise RuntimeError("Reconstruction validation failed; artifacts retained and not registered. Inspect the saved report.")
    active_run.record("reconstruction", finalized_reconstruction_result["report_path"])
    print("Validated reconstruction registered in:", active_run.manifest_path)
else:
    print("Reconstruction disabled; preview only, no PPTX written.")
    from pptx_translation.run_reporting import print_stage_summary
    print_stage_summary(globals().get("active_run"), "reconstruction")


Enable reconstruction to generate a PPTX and validation report, or verify/reuse compatible saved artifacts, without overwriting inputs. Incomplete coverage produces a partial translation. For new runs, passing deliverables are in `final/`; attempt reports and diagnostic PPTX files are under `reports/`. `RUN_SUMMARY.md` links the selected artifacts and reports coverage/status.

Inspect the report and open the output in PowerPoint. Review untranslated/unsupported content, meaning, emphasis and hyperlinks, fonts, clipping, wrapping and right-to-left presentation where relevant. No automatic fitting, geometry, font or RTL adjustments are applied. A passing output can still require linguistic or visual correction; human visual approval is not automatically recorded.


## Operational limits
Ordinary continuation uses the selected manifest and saved stage checkpoints. After an interruption, review saved status and explicitly resume the relevant stage; a request whose response was not durably saved may be repeated. Remove a stale `.lock` only after confirming its writer has stopped.

Specialized offline recovery exists for narrowly eligible historical checkpoints via `active_run.recover_primary()` and `active_run.recover_fallback()`; see [README recovery guidance](README.md#explicit-offline-primary-recovery). These explicit operations revalidate saved responses without provider calls, preserve ancestry and still require semantic evaluation. They are not automatic resume, general checkpoint repair or migration.

A second semantic Claude correction round and visible manual-follow-up slide annotations are not implemented. Resolve pending work and review unresolved cases before treating a translation as complete; always perform human linguistic and PowerPoint inspection before release.
